# 08 · Deep Learning: Rolling Backtest

**Goal:** train the LSTM and the temporal CNN on every fold of the **same rolling chronological backtest** used for the baselines and XGBoost, and save their forecasts.

Each network is trained 3 times per fold with different random starting weights, and the 3 forecasts are averaged. This notebook takes several minutes to run.

In [1]:
import os
import sys

# Find the project folder (the one that contains config.py) and work from there
ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, "config.py")):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
sys.path.append(ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import config

FIG = "reports/figures"
OUT = "data/GeneratedOutput/05.DeepLearning"
os.makedirs(FIG, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

In [2]:
from data_loader import load_raw_data, clean_data
from features import build_features
from backtest import compute_metrics, observed_only
from dl_training import run_dl_backtest

features = build_features(clean_data(load_raw_data(config.RAW_FILE)))
print("Runs per network and fold:", config.N_SEEDS, "| window:", config.SEQ_LEN, "hours | patience:", config.PATIENCE)

Runs per network and fold: 3 | window: 48 hours | patience: 4


## 1. LSTM

In [3]:
lstm_preds = run_dl_backtest("LSTM", features)

  Fold 1: training LSTM


    LSTM run 1/3: 6 epochs, best validation MAE 35.50


    LSTM run 2/3: 8 epochs, best validation MAE 34.99


    LSTM run 3/3: 8 epochs, best validation MAE 35.06
  Fold 1 LSTM: {'MAE': 41.34, 'RMSE': 66.63, 'R2': 0.642}
  Fold 2: training LSTM


    LSTM run 1/3: 9 epochs, best validation MAE 35.19


    LSTM run 2/3: 10 epochs, best validation MAE 35.57


... (3 outputs left out)

    LSTM run 3/3: 14 epochs, best validation MAE 33.18
  Fold 3 LSTM: {'MAE': 34.32, 'RMSE': 54.89, 'R2': 0.684}
  Fold 4: training LSTM


    LSTM run 1/3: 8 epochs, best validation MAE 32.23


    LSTM run 2/3: 11 epochs, best validation MAE 32.20


    LSTM run 3/3: 7 epochs, best validation MAE 32.06
  Fold 4 LSTM: {'MAE': 30.17, 'RMSE': 46.98, 'R2': 0.72}


## 2. Temporal CNN

In [4]:
tcn_preds = run_dl_backtest("TCN", features)

  Fold 1: training TCN


    TCN run 1/3: 13 epochs, best validation MAE 34.50


    TCN run 2/3: 10 epochs, best validation MAE 35.01


    TCN run 3/3: 9 epochs, best validation MAE 34.46


  Fold 1 TCN: {'MAE': 41.07, 'RMSE': 67.26, 'R2': 0.635}
  Fold 2: training TCN


    TCN run 1/3: 15 epochs, best validation MAE 35.18


... (7 outputs left out)

    TCN run 1/3: 18 epochs, best validation MAE 31.67


    TCN run 2/3: 18 epochs, best validation MAE 32.04


    TCN run 3/3: 20 epochs, best validation MAE 31.78


  Fold 4 TCN: {'MAE': 29.86, 'RMSE': 47.04, 'R2': 0.72}


## 3. Save the forecasts

In [5]:
predictions = lstm_preds.copy()
predictions["TCN"] = tcn_preds["TCN"]
predictions.index.name = "forecast_made_at"
predictions.to_csv(f"{OUT}/predictions.csv")

rows = []
for fold, part in observed_only(predictions).groupby("fold"):
    for model in ["LSTM", "TCN"]:
        rows.append({"fold": fold, "model": model, **compute_metrics(part["target"], part[model])})
per_fold = pd.DataFrame(rows)
print("Saved", len(predictions), "forecasts per network")
per_fold.pivot(index="fold", columns="model", values="MAE")

Saved 17514 forecasts per network


model,LSTM,TCN
fold,,
1,41.34,41.07
2,31.28,31.27
3,34.32,34.48
4,30.17,29.86


The next notebook combines these forecasts with XGBoost into the hybrid and compares all models.